## Create Input - Target Pairs

Data(verdict story) -> Tokenize(BPE)  -> Token ID -> input-target pairs

#### 1. data

In [90]:
with open("the-verdict-book.txt","r", encoding="utf-8") as f:
    raw_text=f.read()

print(raw_text[:50])

I HAD always thought Jack Gisburn rather a cheap g


In [91]:
! pip install tiktoken


8574.87s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


#### 2. tokenization(BPE)

In [92]:
import tiktoken
embedding = tiktoken.get_encoding("gpt2")
integers = embedding.encode_ordinary(raw_text[:50])
print("tokens:",    integers)
text = embedding.decode(integers)
print("text:", text)

tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 308]
text: I HAD always thought Jack Gisburn rather a cheap g


#### 3. Token ID

In [93]:
token_id= embedding.encode_ordinary(raw_text)
len(token_id)


5145

#### 4. input-output pair 

In [94]:
stride = 1
context_size = 10
for i in range(0, len(raw_text)//500,context_size):
    input_text = raw_text[i:i+context_size]
    target_text = raw_text[i+1:i+context_size+1]
    input_tokens = embedding.encode_ordinary(input_text)
    target_tokens = embedding.encode_ordinary(target_text)
    print("input:", input_text, "tokens:", input_tokens)
    print("target:", target_text, "tokens:", target_tokens)

input: I HAD alwa tokens: [40, 367, 2885, 435, 10247]
target:  HAD alway tokens: [367, 2885, 435, 1014]
input: ys thought tokens: [893, 1807]
target: s thought  tokens: [82, 1807, 220]
input:  Jack Gisb tokens: [3619, 402, 271, 65]
target: Jack Gisbu tokens: [14295, 402, 271, 11110]
input: urn rather tokens: [700, 2138]
target: rn rather  tokens: [35906, 2138, 220]


##### this is the wrong way bec BPE split the characters into based on total no of words it take as. input if the iput is change then the token_id changes always

In [95]:
context_size = 4
for i in range(0, len(raw_text)//5000):
    input_token = token_id[i:i+context_size]
    target_token = token_id[i+1:i+context_size+1]
    print("input:", input_token)
    print("input text:", embedding.decode_bytes(input_token))
    print("target:    ", target_token)
    print("target text:", embedding.decode_bytes(target_token))

input: [40, 367, 2885, 1464]
input text: b'I HAD always'
target:     [367, 2885, 1464, 1807]
target text: b' HAD always thought'
input: [367, 2885, 1464, 1807]
input text: b' HAD always thought'
target:     [2885, 1464, 1807, 3619]
target text: b'AD always thought Jack'
input: [2885, 1464, 1807, 3619]
input text: b'AD always thought Jack'
target:     [1464, 1807, 3619, 402]
target text: b' always thought Jack G'
input: [1464, 1807, 3619, 402]
input text: b' always thought Jack G'
target:     [1807, 3619, 402, 271]
target text: b' thought Jack Gis'


#### convert into matrix or tensor

In [96]:

batch_size = 3
X=[]
Y=[]
for i in range(0,batch_size):
        input_token = token_id[i:i+context_size]
        X.append(input_token)

        target_token = token_id[i+1:i+context_size+1] ## stride =1 
        Y.append(target_token)
print("X:", X)
print("Y :   ", Y)


X: [[40, 367, 2885, 1464], [367, 2885, 1464, 1807], [2885, 1464, 1807, 3619]]
Y :    [[367, 2885, 1464, 1807], [2885, 1464, 1807, 3619], [1464, 1807, 3619, 402]]


#### introduce the stride 

In [97]:
stride = 2
X=[]
Y=[]
for i in range(0,batch_size):
        input_token = token_id[i*stride:(i*stride)+context_size]
        X.append(input_token)

        target_token = token_id[(i*stride)+1:(i*stride)+context_size+1]  
        Y.append(target_token)
print("X:", X)
print("Y :", Y)

X: [[40, 367, 2885, 1464], [2885, 1464, 1807, 3619], [1807, 3619, 402, 271]]
Y : [[367, 2885, 1464, 1807], [1464, 1807, 3619, 402], [3619, 402, 271, 10899]]


### implement data loader and data sets 


In [98]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text,embedding, context_size, stride):
        self.input_id = []
        self.target_id = []

         # tokenize the entire text 
        token_id= embedding.encode(text)

        # using sliding window approach to create input and target pairs
        for i in range(0, len(token_id) - context_size, stride):
          input_token = token_id[i:i+context_size]
          target_token = token_id[i+1:i+context_size+1]
          self.input_id.append(torch.tensor(input_token))
          self.target_id.append(torch.tensor(target_token))

    def __len__(self):
        return len(self.input_id)

    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]
        input_token = self.token_id[start_idx:start_idx + self.context_size]
        target_token = self.token_id[start_idx + 1:start_idx + self.context_size + 1]
        return input_token, target_token

In [99]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [100]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.10.0
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [101]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [102]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
